# import the libary

In [1]:
from paddleocr import PaddleOCR 
import paddle
import os

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# device used

In [2]:
# paddle.device.get_device()

In [3]:
# paddle.device.cuda.device_count()

# loading image path

In [88]:
i="4"
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/4.jpg'

# read img

In [89]:
print(os.path.exists(image_path))

True


In [90]:
img = cv2.imread(image_path)

# run paddleocr on img 

In [ ]:
if (os.path.exists(image_path) == False):
    print(f"{image_path} Path not found
if (img is None) :
    print(f"{image_path} Image not found")
else:
    # setting tread 
    # os.environ["OMP_NUM_THREADS"] = "8"
    # os.environ["MKL_NUM_THREADS"] = "8"
    # applying ocr on the image
    ocr = PaddleOCR(use_doc_orientation_classify=False, 
        use_doc_unwarping=False, 
        use_textline_orientation=False,
        lang='en',
        device=paddle.device.get_device(),
        cpu_threads=8 
       )
    result = ocr.predict(img)

Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


In [ ]:
result

In [4]:
import os
import cv2
import paddle
from paddleocr import PaddleOCR


class OCREngine:

    def __init__(self):

        self.ocr = PaddleOCR(
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False,
            lang='en',
            device=paddle.device.get_device(),
            cpu_threads=8
        )

    def run_ocr(self, image_path):

        if not os.path.exists(image_path):
            print(f"{image_path} Path not found")
            return None

        img = cv2.imread(image_path)

        if img is None:
            print(f"{image_path} Image not found")
            return None

        result = self.ocr.predict(img)

        return result
        
    def downscaling(self, poly_list):

        poly_downscaling = []

        for poly in poly_list:

            # convert to float
            poly = np.array(poly, dtype=np.float32)

            # draw original polygon
            cv2.polylines(
                output,
                [np.array(poly, dtype=np.int32)],
                isClosed=True,
                color=(255, 0, 0),
                thickness=2
            )

            # bounding box
            x_min, y_min = np.min(poly, axis=0)
            x_max, y_max = np.max(poly, axis=0)

            w = x_max - x_min
            h = y_max - y_min

            # adaptive shrink
            shrink_factor_h = max(0.6, min(0.9, 1 - 20 / w))
            shrink_factor_v = max(0.6, min(0.9, 1 - 20 / h))

            # center
            center_x = np.mean(poly[:, 0])
            center_y = np.mean(poly[:, 1])

            poly_box = []

            for (x, y) in poly:

                x_new = int(
                    center_x + (x - center_x) * shrink_factor_h
                )

                y_new = int(
                    center_y + (y - center_y) * shrink_factor_v
                )

                poly_box.append([x_new, y_new])

            poly_downscaling.append(
                np.array(poly_box, dtype=np.int16)
            )

        return poly_downscaling

    def downscaling_poly_ocr_result(self, result):

        # downscale polygons
        downscaling_poly = self.downscaling(
            result[0]['dt_polys']
        )
        
        # combine polygon + text + score
        ocr_result = [
            (poly, text, scores)
            for poly, text, scores in zip(
                downscaling_poly,
                result[0]['rec_texts'],
                result[0]['rec_scores']
            )
        ]
        
        return ocr_result

In [5]:
ocr = OCREngine()

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.
